In [9]:
import os 
os.environ['JAX_JIT_PJIT_API_MERGE'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '0'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '.90'
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update('jax_platform_name', 'gpu')
from qiskit_dynamics.array import Array
Array.set_default_backend('jax')

# Define undriven system using scqubits, then convert to qiskit

In [10]:
from utils import *
import numpy as np
from qiskit.quantum_info import Operator
from qiskit_dynamics import Solver
import jax.numpy as jnp
from qiskit import pulse

EJ=8.9
EC=2.5
EL=0.5
g_strength = 0.3
E_osc = 3
qubit_level = 6
osc_level = 15


qbt = scqubits.Fluxonium(EJ=EJ,EC=EC,EL=EL,flux=0,cutoff=30,truncated_dim=qubit_level)
osc = scqubits.Oscillator(E_osc=E_osc,truncated_dim=osc_level)
hilbertspace = scqubits.HilbertSpace([qbt, osc])
hilbertspace.add_interaction(g_strength=g_strength,op1=qbt.n_operator,op2=osc.creation_operator,add_hc=True)
hilbertspace.generate_lookup()
product_to_dressed = generate_single_mapping(hilbertspace.hamiltonian())
# plot_specturum(qbt, osc, hilbertspace)

In [11]:



def qobj_to_Array(matrix):
    if type(matrix) == qutip.qobj.Qobj:
        matrix = matrix.full()
    # return Operator(matrix, input_dims = (matrix.shape[0],),output_dims = (matrix.shape[1],))
    return Array(matrix)

(evals,) = hilbertspace["evals"]
diag_matrix = np.diag(evals)
static_hamiltonian = 2 * np.pi * qobj_to_Array(diag_matrix)


In [12]:
leakage_dressed_state_osc_0 = product_to_dressed[(0,0)]
leakage_dressed_state_osc_1 = product_to_dressed[(0,1)]

a = 2 * np.pi * hilbertspace.op_in_dressed_eigenbasis(op=osc.annihilation_operator)


tot_dims = a.shape[0]
g0 = jnp.zeros(tot_dims).at[product_to_dressed[(0, 0)]].set(1).reshape(-1, 1)
e0 = jnp.zeros(tot_dims).at[product_to_dressed[(1, 0)]].set(1).reshape(-1, 1)
f0 = jnp.zeros(tot_dims).at[product_to_dressed[(2, 0)]].set(1).reshape(-1, 1)
h0 = jnp.zeros(tot_dims).at[product_to_dressed[(3, 0)]].set(1).reshape(-1, 1)

pn_op = jnp.array((a.dag()*a).full())
a_op = jnp.array(a.full())

# tot_time = 500
# matrix_element_driven = abs((a+a.dag()).data.toarray()[leakage_dressed_state_osc_0][leakage_dressed_state_osc_1])
# base_drive_amplitude = 1/tot_time
# base_drive_amplitude = base_drive_amplitude/matrix_element_driven


driven_operator =  qobj_to_Array(a+a.dag())

In [13]:
# from qiskit_dynamics.pulse import InstructionToSignals
# converter = InstructionToSignals(signal_sample_dt, carriers={"d0": carrier_freq})
# signals = converter.get_signals(square)
# fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
# for ax, title in zip(axs, ["envelope", "signal"]):
#     signals[0].draw(0, tot_time/2000, 2000, title, axis=ax)
#     ax.set_xlabel("Time (ns)")
#     ax.set_ylabel("Amplitude")
#     ax.set_title(title)


In [14]:
tot_time = 10
tlist =  jnp.linspace(0,tot_time, int(tot_time))
w_d = transition_frequency(hilbertspace,leakage_dressed_state_osc_0,leakage_dressed_state_osc_1 )
carrier_freq = w_d * 2 * np.pi

signal_sample_dt = 0.01 # Sample rate of the backend in ns.

base_drive_amplitude = 0.03 * 2 * np.pi

with pulse.build(name="square") as square:
    pulse.play(pulse.Constant(duration = int(tot_time/signal_sample_dt), amp = base_drive_amplitude), pulse.DriveChannel(0))

# square.draw()

kappa = 0.5*np.pi/75
L_ops = [np.sqrt(kappa)*qobj_to_Array(a)/10]


ham_solver =  Solver(
                hamiltonian_operators=[driven_operator],
                static_hamiltonian=static_hamiltonian,
                static_dissipators=L_ops,
                # rotating_frame=static_hamiltonian,
                hamiltonian_channels=['d0'],
                channel_carrier_freqs={'d0': carrier_freq},
                dt=signal_sample_dt
            )


In [20]:
ham_solver=ham_solver
y0=g0
tlist=tlist
print(f"tlist: {tlist}")
signals=square
chunk_size=0.001
ham_solver.model.evaluation_mode = "sparse_vectorized"
if y0.shape[0] != ham_solver.model.dim**2:
    y0 = y0 @ y0.conj().T
    y0 = y0.flatten(order='F')

ode_result = CustomOdeResult()


current_state = y0
chunk_results = []
t_results = []
total_time = tlist[-1] - tlist[0]
num_intervals = int(len(tlist) / chunk_size)
interval_length = total_time / num_intervals
print(f"total_time: {total_time}")
print(f"num_intervals: {num_intervals}")
print(f"interval_length: {interval_length}")

max_dt = interval_length

for i in range(num_intervals):
    t_start = tlist[0] + i * interval_length
    t_end = t_start + interval_length
    t_eval_interval = jnp.array([t_end])
    print(f"t_eval_interval {t_eval_interval}")
    print(f"[t_start, t_end]: {[t_start, t_end]}")
    result = ham_solver.solve(
        y0=current_state,
        t_span=[t_start, t_end],
        signals=signals,
        method='jax_expm_parallel',
        t_eval=t_eval_interval,
        max_dt=max_dt
    )

    chunk_results.append(result.y[-1])
    t_results.append(t_eval_interval[-1])

    current_state = result.y[-1]
    clear_output(wait=True)
    print(f"Progress: Interval {i + 1}/{num_intervals} solved.")
ode_result = CustomOdeResult(t=t_results, y=chunk_results)

tlist: [ 0.          1.11111111  2.22222222  3.33333333  4.44444444  5.55555556
  6.66666667  7.77777778  8.88888889 10.        ]


2023-08-25 19:01:32.706919: W external/tsl/tsl/framework/bfc_allocator.cc:485] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1001.13MiB (rounded to 1049760000)requested by op 
2023-08-25 19:01:32.706964: I external/tsl/tsl/framework/bfc_allocator.cc:1039] BFCAllocator dump for GPU_0_bfc
2023-08-25 19:01:32.706970: I external/tsl/tsl/framework/bfc_allocator.cc:1046] Bin (256): 	Total Chunks: 10, Chunks in use: 10. 2.5KiB allocated for chunks. 2.5KiB in use in bin. 152B client-requested in use in bin.
2023-08-25 19:01:32.706972: I external/tsl/tsl/framework/bfc_allocator.cc:1046] Bin (512): 	Total Chunks: 3, Chunks in use: 3. 2.2KiB allocated for chunks. 2.2KiB in use in bin. 2.1KiB client-requested in use in bin.
2023-08-25 19:01:32.706974: I external/tsl/tsl/framework/bfc_allocator.cc:1046] Bin (1024): 	Total Chunks: 2, Chunks in use: 2. 2.0KiB allocated for chunks. 2.0KiB in use in bin. 1.4KiB client-requested in use in bin.
2023-08-25 19:01:32.706976: I external/tsl/tsl/

ValueError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 1049760000 bytes.
BufferAssignment OOM Debugging.
BufferAssignment stats:
             parameter allocation:   189.8KiB
              constant allocation:         0B
        maybe_live_out allocation: 1001.13MiB
     preallocated temp allocation:         0B
                 total allocation: 1001.31MiB
              total fragmentation:         0B (0.00%)
Peak buffers:
	Buffer 1:
		Size: 1001.13MiB
		Operator: op_name="jit(kron)/jit(main)/reshape[new_sizes=(8100, 8100) dimensions=None]" source_file="/home/jiakai/.local/lib/python3.10/site-packages/qiskit_dynamics/array/array.py" source_line=290
		XLA Label: fusion
		Shape: c128[8100,8100]
		==========================

	Buffer 2:
		Size: 126.6KiB
		Entry Parameter Subshape: c128[90,90]
		==========================

	Buffer 3:
		Size: 63.3KiB
		Entry Parameter Subshape: f64[90,90]
		==========================



In [8]:

results = []
for psi0 in [g0,e0,f0,h0]:
    result =solve_with_jax_gpu_lindbladian(ham_solver=ham_solver,
                    y0=psi0,
                    tlist=tlist,
                    signals=square,
                    chunk_size=0.01
                )
    results.append(result)

TypeError: iteration over a 0-d array

In [ ]:
plot_population(results,qubit_level,osc_level,product_to_dressed,a,w_d,tlist,fourier=True)

TypeError: dot_general requires contracting dimensions to have the same shape, got (16,) and (256,).